In [ ]:
import sys
path_to_pip_installs = "/tmp/test_env"
if path_to_pip_installs not in sys.path:
    sys.path.insert(0, path_to_pip_installs)

path = "/home/students/studweilc1/MU-Diff/"
# add this to path for imports
if path not in sys.path:
    sys.path.append(path)

In [16]:
import numpy as np

input_path = "../data/my_data2"
contrasts = ["T1_mapping_fl2d", "BOLD", "Diffusion", "DIXON"]
split="val"

masks_T1 = np.load(f"{input_path}/{split}/T1_mapping_fl2d_masks.npy")
masks_BOLD = np.load(f"{input_path}/{split}/BOLD_masks.npy")
masks_Diffusion = np.load(f"{input_path}/{split}/Diffusion_masks.npy")
masks_DIXON = np.load(f"{input_path}/{split}/DIXON_masks.npy")

In [34]:
def dice_coefficient_numpy(y_true, y_pred, smooth=1e-6):
    y_true_f = np.reshape(y_true, [-1])
    y_pred_f = np.reshape(y_pred, [-1])
    intersection = np.sum(y_true_f * y_pred_f)
    return (2. * intersection + smooth) / (np.sum(y_true_f) + np.sum(y_pred_f) + smooth)

def calculate_dice_score_with_shift(moving_mask, fixed_mask, shift_x, shift_y):
    # Shift the moving mask
    shifted_moving_mask = np.roll(moving_mask, shift=(shift_x, shift_y), axis=(0, 1))
    # Calculate Dice score
    dice_score = dice_coefficient_numpy(fixed_mask, shifted_moving_mask)
    return dice_score

def calculate_max_dice(moving_mask, fixed_mask, max_shift=5):
    best_dice = 0
    best_shift = (0, 0)
    for shift_x in range(-max_shift, max_shift + 1):
        for shift_y in range(-max_shift, max_shift + 1):
            dice_score = calculate_dice_score_with_shift(moving_mask, fixed_mask, shift_x, shift_y)
            if dice_score > best_dice:
                best_dice = dice_score
                best_shift = (shift_x, shift_y)
    return best_dice, best_shift



In [56]:
import pandas as pd

res_df = pd.DataFrame(columns=["index","org_dice", "best_dice", "best_shift", "improvement", "mask_diff"])

for i in range(len(masks_T1)):
    mask1 = masks_T1[i]
    mask2 = masks_DIXON[i]

    mask_1_size = np.sum(mask1)
    mask_2_size = np.sum(mask2)
    mask_diff = abs(mask_1_size - mask_2_size) / max(mask_1_size, mask_2_size)

    org_dice = dice_coefficient_numpy(mask1, mask2)
    best_dice, best_shift = calculate_max_dice(mask1, mask2, max_shift=5)

    if org_dice < best_dice:
        imp = best_dice - org_dice
    else:
        imp = 0.0
    res_df = pd.concat([res_df, pd.DataFrame({"index": [i], "org_dice": [org_dice], "best_dice": [best_dice], "best_shift": [best_shift], "improvement": [imp], "mask_diff": [mask_diff]})], ignore_index=True)
display(res_df[["org_dice", "best_dice", "improvement", "mask_diff"]].mean())
display(res_df[res_df["mask_diff"]<0.1][["org_dice", "best_dice", "improvement", "mask_diff"]].mean())


The behavior of DataFrame concatenation with empty or all-NA entries is deprecated. In a future version, this will no longer exclude empty or all-NA columns when determining the result dtypes. To retain the old behavior, exclude the relevant entries before the concat operation.


org_dice       0.812496
best_dice      0.855430
improvement    0.042934
mask_diff      0.111887
dtype: float64

org_dice       0.839030
best_dice      0.888036
improvement    0.049006
mask_diff      0.047214
dtype: float64

In [58]:
import pandas as pd

res_df = pd.DataFrame(columns=["index","org_dice", "best_dice", "best_shift", "improvement", "mask_diff"])

for i in range(len(masks_T1)):
    mask1 = masks_BOLD[i]
    mask2 = masks_DIXON[i]

    mask_1_size = np.sum(mask1)
    mask_2_size = np.sum(mask2)
    mask_diff = abs(mask_1_size - mask_2_size) / max(mask_1_size, mask_2_size)

    org_dice = dice_coefficient_numpy(mask1, mask2)
    best_dice, best_shift = calculate_max_dice(mask1, mask2, max_shift=5)

    if org_dice < best_dice:
        imp = best_dice - org_dice
    else:
        imp = 0.0
    res_df = pd.concat([res_df, pd.DataFrame({"index": [i], "org_dice": [org_dice], "best_dice": [best_dice], "best_shift": [best_shift], "improvement": [imp], "mask_diff": [mask_diff]})], ignore_index=True)
display(res_df[["org_dice", "best_dice", "improvement", "mask_diff"]].mean())
display(res_df[res_df["mask_diff"]<0.1][["org_dice", "best_dice", "improvement", "mask_diff"]].mean())


The behavior of DataFrame concatenation with empty or all-NA entries is deprecated. In a future version, this will no longer exclude empty or all-NA columns when determining the result dtypes. To retain the old behavior, exclude the relevant entries before the concat operation.


org_dice       0.787839
best_dice      0.807026
improvement    0.019188
mask_diff      0.136872
dtype: float64

org_dice       0.825620
best_dice      0.845368
improvement    0.019748
mask_diff      0.046686
dtype: float64

In [59]:
import pandas as pd

res_df = pd.DataFrame(columns=["index","org_dice", "best_dice", "best_shift", "improvement", "mask_diff"])

for i in range(len(masks_T1)):
    mask1 = masks_Diffusion[i]
    mask2 = masks_DIXON[i]

    mask_1_size = np.sum(mask1)
    mask_2_size = np.sum(mask2)
    mask_diff = abs(mask_1_size - mask_2_size) / max(mask_1_size, mask_2_size)

    org_dice = dice_coefficient_numpy(mask1, mask2)
    best_dice, best_shift = calculate_max_dice(mask1, mask2, max_shift=5)

    if org_dice < best_dice:
        imp = best_dice - org_dice
    else:
        imp = 0.0
    res_df = pd.concat([res_df, pd.DataFrame({"index": [i], "org_dice": [org_dice], "best_dice": [best_dice], "best_shift": [best_shift], "improvement": [imp], "mask_diff": [mask_diff]})], ignore_index=True)
display(res_df[["org_dice", "best_dice", "improvement", "mask_diff"]].mean())
display(res_df[res_df["mask_diff"]<0.1][["org_dice", "best_dice", "improvement", "mask_diff"]].mean())


The behavior of DataFrame concatenation with empty or all-NA entries is deprecated. In a future version, this will no longer exclude empty or all-NA columns when determining the result dtypes. To retain the old behavior, exclude the relevant entries before the concat operation.


org_dice       0.756333
best_dice      0.819416
improvement    0.063083
mask_diff      0.099388
dtype: float64

org_dice       0.774891
best_dice      0.840053
improvement    0.065161
mask_diff      0.041723
dtype: float64